<a href="https://colab.research.google.com/github/Munjiwon/SpecialTopics-in-TextMining/blob/master/ch02/bow_tfidf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2주차 실습 1 — BoW와 TF-IDF 비교

**이 노트북의 새 개념**: 같은 문서 집합에 카운트(BoW)와 TF-IDF 가중치를 매겨 상위 단어가 어떻게 달라지는지 본다.

- tf(w, d) = 1 + log c(w, d),  idf(w) = log N / df(w)
- scikit-learn 의 기본 idf 는 log((1+N)/(1+df)) + 1 이다(스무딩) — 교과서 식과 값이 다르다.

In [ ]:
import sys, sklearn, torch
print("Python", sys.version.split()[0], "| scikit-learn", sklearn.__version__, "| torch", torch.__version__)

Python 3.13.15 | scikit-learn 1.6.1 | torch 2.11.0+cpu


In [ ]:
# 말뭉치 올리기 — 1주차와 같은 파일을 쓴다(빈 줄로 구분된 문단 하나를 문서 하나로 본다)
CORPUS_PATH = "/content/corpus.txt"      # Colab 밖에서 실행할 때는 이 경로를 직접 바꾼다
try:
    from google.colab import files
    uploaded = files.upload()
    CORPUS_PATH = "/content/" + next(iter(uploaded))
except ImportError:
    pass

with open(CORPUS_PATH, encoding="utf-8") as f:
    raw = f.read()
docs = [d.strip() for d in raw.split("\n\n") if len(d.strip()) > 20]   # 문단 = 문서
print(f"문서 {len(docs):,}개, 예시: {docs[0][:60]}...")

Saving corpus_100k_tokens.txt to corpus_100k_tokens.txt
문서 1개, 예시: 0	노래가 너무 적음
0	돌겠네 진짜. 황숙아, 어크 공장 그만 돌려라. 죽는다.
1	막노동 체험판 막노동 ...


## 벡터화
토큰은 공백 기준(형태소 기반 비교는 `korean_tokenize.ipynb`).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

count_vec = CountVectorizer(token_pattern=r"\S+")               # 공백으로 나뉜 덩어리 하나 = 토큰
tfidf_vec = TfidfVectorizer(token_pattern=r"\S+", sublinear_tf=True)   # sublinear_tf: 1 + log c
X_count = count_vec.fit_transform(docs)                          # 희소 행렬 (문서 수 × 어휘 크기)
X_tfidf = tfidf_vec.fit_transform(docs)
vocab = count_vec.get_feature_names_out()
print("행렬 크기:", X_count.shape, " 0이 아닌 칸 비율:", f"{X_count.nnz / (X_count.shape[0]*X_count.shape[1]):.4%}")

행렬 크기: (1, 283161)  0이 아닌 칸 비율: 100.0000%


## 첫 문서의 상위 단어 비교
카운트 쪽에는 기능어가, TF-IDF 쪽에는 그 문서를 특징짓는 단어가 올라온다.

In [ ]:
import numpy as np
d = 0
top_count = np.argsort(-X_count[d].toarray()[0])[:8]
top_tfidf = np.argsort(-X_tfidf[d].toarray()[0])[:8]
print("Count :", [vocab[i] for i in top_count])
print("TF-IDF:", [tfidf_vec.get_feature_names_out()[i] for i in top_tfidf])

Count : ['1', '0', '게임', '너무', '이', '그냥', '게임을', '수']
TF-IDF: ['1', '0', '게임', '너무', '이', '그냥', '게임을', '수']


## idf 손계산과 대조
`smooth_idf=True`(기본값)일 때 idf = ln((1+N)/(1+df)) + 1 이다.

In [ ]:
import math
N = len(docs)
df = np.asarray((X_count > 0).sum(axis=0)).ravel()               # 단어마다 등장한 문서 수
for w in ["의", "는", vocab[top_tfidf[0]]]:
    if w in tfidf_vec.vocabulary_:
        j = count_vec.vocabulary_[w]
        by_hand = math.log((1 + N) / (1 + df[j])) + 1
        print(f"{w!r:>8}  df={df[j]:>5}  손계산 idf={by_hand:.4f}  sklearn idf={tfidf_vec.idf_[tfidf_vec.vocabulary_[w]]:.4f}")

     '의'  df=    1  손계산 idf=1.0000  sklearn idf=1.0000
     '는'  df=    1  손계산 idf=1.0000  sklearn idf=1.0000
     '1'  df=    1  손계산 idf=1.0000  sklearn idf=1.0000


## 코사인 유사도 — PyTorch 텐서로
TF-IDF 행은 이미 L2 정규화되어 있으므로 내적이 곧 코사인 유사도다.

In [ ]:
T = torch.tensor(X_tfidf[:200].toarray(), dtype=torch.float32)   # 앞 200개 문서만(메모리 절약)
sim = T @ T.T                                                     # (200 × 200) 코사인 유사도
sim.fill_diagonal_(-1)                                            # 자기 자신은 제외
i, j = divmod(int(sim.argmax()), sim.shape[1])
print(f"가장 비슷한 문서 쌍: {i}, {j}  유사도={sim[i, j]:.3f}")
print(" -", docs[i][:60]); print(" -", docs[j][:60])

가장 비슷한 문서 쌍: 0, 0  유사도=-1.000
 - 0	노래가 너무 적음
0	돌겠네 진짜. 황숙아, 어크 공장 그만 돌려라. 죽는다.
1	막노동 체험판 막노동 
 - 0	노래가 너무 적음
0	돌겠네 진짜. 황숙아, 어크 공장 그만 돌려라. 죽는다.
1	막노동 체험판 막노동 


## 직접 해 보기
1. `sublinear_tf=False` 로 바꾸면 상위 단어가 어떻게 바뀌는가?
2. `smooth_idf=False` 로 두고 교과서 식 log(N/df) + 1 과 비교해 보자.
3. 불용어 목록을 `stop_words=[...]` 로 넣으면 Count 상위어가 어떻게 달라지는가?